In [8]:
import json
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import warnings
warnings.filterwarnings('ignore')

In [9]:
THRESHOLDS = [ 10, 20]

MP_JOINTS = [
    "WRIST", "THUMB_CMC", "THUMB_MCP", "THUMB_IP", "THUMB_TIP",
    "INDEX_FINGER_MCP", "INDEX_FINGER_PIP", "INDEX_FINGER_DIP", "INDEX_FINGER_TIP",
    "MIDDLE_FINGER_MCP", "MIDDLE_FINGER_PIP", "MIDDLE_FINGER_DIP", "MIDDLE_FINGER_TIP",
    "RING_FINGER_MCP", "RING_FINGER_PIP", "RING_FINGER_DIP", "RING_FINGER_TIP",
    "PINKY_MCP", "PINKY_PIP", "PINKY_DIP", "PINKY_TIP"
]

JOINT_TO_IDX = {name: i for i, name in enumerate(MP_JOINTS)}

FINGER_GROUPS = {
    "thumb": MP_JOINTS[1:5],
    "index": MP_JOINTS[5:9],
    "middle": MP_JOINTS[9:13],
    "ring": MP_JOINTS[13:17],
    "pinky": MP_JOINTS[17:21]
}

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20)
]

In [10]:
def draw_skeleton(ax, joints, connections, color='green', marker='o', label=None):
    xs, ys = [], []
    for pt in joints.values():
        xs.append(pt[0])
        ys.append(pt[1])

    ax.scatter(xs, ys, c=color, marker=marker, label=label, s=40, edgecolors='black')

    for s, e in connections:
        s_name, e_name = MP_JOINTS[s], MP_JOINTS[e]
        if s_name in joints and e_name in joints:
            ax.plot([joints[s_name][0], joints[e_name][0]],
                    [joints[s_name][1], joints[e_name][1]],
                    c=color, linewidth=2)

In [11]:
def init_global_stats():
    return {
        "total_joints": 0,
        "correct": {t: 0 for t in THRESHOLDS},
        "all_errors": [],
        "per_joint": {j: {"total":0, "correct":0} for j in MP_JOINTS}
    }

In [12]:
def run_analysis(batch_path, stats, global_mistakes):
    ERROR_JSON = os.path.join(batch_path, 'pixel_errors_by_joint.json')
    MP_JSON = os.path.join(batch_path, 'mediapipe_landmarks.json')
    XML_FILE = os.path.join(batch_path, 'annotations.xml')
    IMAGE_DIR = os.path.join(batch_path, 'frames')

    GLOBAL_IMG_ROOT = "mistake_images"
    batch_name = os.path.basename(batch_path)
    SAVE_DIR = os.path.join(GLOBAL_IMG_ROOT, batch_name)
    os.makedirs(SAVE_DIR, exist_ok=True)

    with open(ERROR_JSON) as f:
        error_data = json.load(f)
    with open(MP_JSON) as f:
        mp_data = json.load(f)

    root = ET.parse(XML_FILE).getroot()

    for image_tag in root.findall('image'):
        frame_name = os.path.basename(image_tag.get('name'))
        if frame_name not in error_data:
            continue

        width = int(image_tag.get('width'))
        height = int(image_tag.get('height'))

        img = cv2.imread(os.path.join(IMAGE_DIR, frame_name))
        if img is None:
            img = np.zeros((height, width, 3), dtype=np.uint8)

        fig, ax = plt.subplots(figsize=(width/100, height/100), dpi=100)
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.axis('off')

        frame_has_error = False

        for skeleton in image_tag.findall('skeleton'):
            hand = skeleton.get('label')
            if hand not in error_data[frame_name]:
                continue

            gt_bad, mp_bad = {}, {}

            for p in skeleton.findall('points'):
                joint = p.get('label')
                if joint not in error_data[frame_name][hand]:
                    continue

                error = error_data[frame_name][hand][joint]

                stats["total_joints"] += 1
                stats["all_errors"].append(error)
                stats["per_joint"][joint]["total"] += 1

                for t in THRESHOLDS:
                    if error <= t:
                        stats["correct"][t] += 1

                if error <= 10:
                    stats["per_joint"][joint]["correct"] += 1

                gx, gy = map(float, p.get('points').split(','))
                idx = JOINT_TO_IDX[joint]
                mp_pt = mp_data[frame_name][hand][idx]
                mx, my = mp_pt['x']*width, mp_pt['y']*height

                if error > 10:
                    frame_has_error = True

                    if frame_name not in global_mistakes:
                        global_mistakes[frame_name] = {}
                    if hand not in global_mistakes[frame_name]:
                        global_mistakes[frame_name][hand] = {}

                    global_mistakes[frame_name][hand][joint] = {
                        "error": error,
                        "gt": [gx, gy],
                        "pred": [mx, my]
                    }

                    gt_bad[joint] = (gx, gy)
                    mp_bad[joint] = (mx, my)

            if gt_bad and mp_bad:
                draw_skeleton(ax, gt_bad, HAND_CONNECTIONS, 'green', 'o', f'GT {hand}')
                draw_skeleton(ax, mp_bad, HAND_CONNECTIONS, 'red', 'x', f'MP {hand}')

                for j in gt_bad:
                    if j in mp_bad:
                        ax.plot([gt_bad[j][0], mp_bad[j][0]],
                                [gt_bad[j][1], mp_bad[j][1]],
                                c='blue', linewidth=1)

        if frame_has_error:
            plt.tight_layout()
            plt.savefig(os.path.join(SAVE_DIR, frame_name))

        plt.close(fig)

In [13]:
def compute_final_metrics(stats):
    total = stats["total_joints"]
    errors = np.array(stats["all_errors"])

    metrics = {
        "MPJPE": float(np.mean(errors)) if len(errors) else 0,
        "total_joints": total
    }

    #  Add correct counts + accuracy
    for t in THRESHOLDS:
        correct = stats["correct"][t]
        metrics[f"PCK@{t}"] = {
            "correct": correct,
            "total": total,
            "accuracy": correct / total if total else 0
        }

    # per joint accuracy
    per_joint_acc = {}
    for j, d in stats["per_joint"].items():
        if d["total"] > 0:
            per_joint_acc[j] = {
                "correct": d["correct"],
                "total": d["total"],
                "accuracy": d["correct"] / d["total"]
            }
    metrics["per_joint_accuracy"] = per_joint_acc

    # per finger
    finger_acc = {}
    for f, joints in FINGER_GROUPS.items():
        vals = [
            d["correct"] / d["total"]
            for j, d in stats["per_joint"].items()
            if j in joints and d["total"] > 0
        ]
        if vals:
            finger_acc[f] = float(np.mean(vals))
    metrics["per_finger_accuracy"] = finger_acc

    # distribution
    metrics["error_distribution"] = {
        "5-10": float(np.mean((errors >= 5) & (errors < 10))),
        ">10": float(np.mean(errors >= 10))
    }

    return metrics

In [14]:
if __name__ == "__main__":
    #batches = [f for f in os.listdir(".") if f.startswith("frames-batch-")]
    batches=[1,2]
    stats = init_global_stats()
    global_mistakes = {}

    for b in sorted(batches):
        print("Processing", b)
        run_analysis(b, stats, global_mistakes)

    final_metrics = compute_final_metrics(stats)

    with open("global_metrics_summary.json", "w") as f:
        json.dump(final_metrics, f, indent=4)

    with open("global_mistakes_detailed.json", "w") as f:
        json.dump(global_mistakes, f, indent=4)

    print("\nDONE")

Processing 1


TypeError: expected str, bytes or os.PathLike object, not int